# Data Merge Tutorial 2: Fuzzy Matching Unresolved KC Building Datasets  
### Case Study: Resolving Remaining KC Building Records Across Files

## Introduction

In this tutorial series, we continue the Kansas City building merge workflow by focusing on the unresolved records that remain after the exact matching in turorial 1.

Real-world administrative datasets are rarely ready to merge perfectly. Even after standardizing fields and joining obvious exact matches, some records still differ because of abbreviations, spelling changes, naming variations, or slightly different address formatting. That is where fuzzy matching becomes useful.

## Tutorial Plan

We will:

1. Load the outputs from tutorial 1 (matched and unmatched dataframes),
3. use fuzzy matching only on that reduced set,
4. and create a setup for manual inspection.


## Core Python Libraries

- **pandas**: table wrangling and joins  
- **numpy**: missing values and simple numeric logic  
- **re**: text cleanup for IDs and addresses  
- **rapidfuzz**: fuzzy similarity scoring for unresolved pairs
- **ipywidgets**: Interactive widget for manual review of fuzzy matching


In [70]:
import re
import numpy as np
import pandas as pd
import os

try:
    from rapidfuzz import fuzz
except ImportError:
    raise ImportError("Install rapidfuzz first: pip install rapidfuzz")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


## Data configuration

By default this notebook uses a small toy example so the workflow is easy to teach.

To adapt it to the real files:

1. set `USE_TOY_DATA = False`
2. point the file paths to your 2025 and 2026 exports
3. update the column names in the standardization cells if needed




The toy data below mirrors the kinds of differences that show up across years:

- slightly different building names,
- slightly different address formatting,
- missing IDs in some rows,
- and one clearly unresolved case.

This makes it easier to teach the join logic before applying it to the full files.


In [71]:
USE_TOY_DATA = True

toy_2025 = pd.DataFrame({
    "Benchmarking ID": ["1001", "1002", np.nan, np.nan],
    "Building Name": ["Main Plaza", "River Center", "Oak Tower", "West Campus Annex"],
    "Building Address": ["100 Main Street", "200 River Rd", "3300 Oak Ave", "900 West Campus Dr"],
    "Building City": ["Kansas City", "Kansas City", "Kansas City", "Kansas City"],
    "Building State": ["MO", "MO", "MO", "MO"],
    "Building Zip": ["64105", "64106", "64111", "64114"]
})

toy_2026 = pd.DataFrame({
    "Benchmarking ID": ["1001", np.nan, np.nan, np.nan],
    "Property Name": ["Main Plaza Office", "River Center", "Oak TWR", "South Campus Annex"],
    "Address 1": ["100 MAIN ST.", "200 River Road", "3300 Oak Avenue", "910 West Campus Drive"],
    "City": ["Kansas City", "Kansas City", "Kansas City", "Kansas City"],
    "State/Province": ["MO", "MO", "MO", "MO"],
    "Postal Code": ["64105", "64106", "64111", "64114"]
})

display(toy_2025)
display(toy_2026)

raw_2025 = toy_2025.copy()
raw_2026 = toy_2026.copy()

,Benchmarking ID,Building Name,Building Address,Building City,Building State,Building Zip
0,1001,Main Plaza,100 Main Street,Kansas City,MO,64105
1,1002,River Center,200 River Rd,Kansas City,MO,64106
2,NaN,Oak Tower,3300 Oak Ave,Kansas City,MO,64111
3,NaN,West Campus Annex,900 West Campus Dr,Kansas City,MO,64114


,Benchmarking ID,Property Name,Address 1,City,State/Province,Postal Code
0,1001,Main Plaza Office,100 MAIN ST.,Kansas City,MO,64105
1,NaN,River Center,200 River Road,Kansas City,MO,64106
2,NaN,Oak TWR,3300 Oak Avenue,Kansas City,MO,64111
3,NaN,South Campus Annex,910 West Campus Drive,Kansas City,MO,64114


In [72]:
display(raw_2025.head())
display(raw_2026.head())


,Benchmarking ID,Building Name,Building Address,Building City,Building State,Building Zip
0,1001,Main Plaza,100 Main Street,Kansas City,MO,64105
1,1002,River Center,200 River Rd,Kansas City,MO,64106
2,NaN,Oak Tower,3300 Oak Ave,Kansas City,MO,64111
3,NaN,West Campus Annex,900 West Campus Dr,Kansas City,MO,64114


,Benchmarking ID,Property Name,Address 1,City,State/Province,Postal Code
0,1001,Main Plaza Office,100 MAIN ST.,Kansas City,MO,64105
1,NaN,River Center,200 River Road,Kansas City,MO,64106
2,NaN,Oak TWR,3300 Oak Avenue,Kansas City,MO,64111
3,NaN,South Campus Annex,910 West Campus Drive,Kansas City,MO,64114


Looking at these dataframes, some headers have different naming convention (ie. 2025["Building Address"] and 2026["Address 1"]). To merge we will need to identify which headers we want from.
Benchmarking ID and address will be a good place to start, we can further use building name as a unique identifyer.

In [73]:
# Setup dataframes with consistent column names and formats for matching
# # We make a new dataframe using dataframe.copy() and pull/rename the headers we want into a consistent format. 
# # # This way we can keep the original raw dataframes intact for reference.
df2025 = raw_2025.rename(columns={
    "Benchmarking ID": "benchmark_id",
    "Building Name": "building_name",
    "Building Address": "address",
    "Building City": "city",
    "Building State": "state",
    "Building Zip": "zip"
}).copy()

df2026 = raw_2026.rename(columns={
    "Benchmarking ID": "benchmark_id",
    "Property Name": "building_name",
    "Address 1": "address",
    "City": "city",
    "State/Province": "state",
    "Postal Code": "zip"
}).copy()

keep_cols = ["benchmark_id", "building_name", "address", "city", "state", "zip"]

df2025 = df2025[keep_cols].copy()
df2026 = df2026[keep_cols].copy()

# Add reporting year and source record ID for traceability
df2025["reporting_year"] = 2025
df2026["reporting_year"] = 2026

df2025["source_record_id"] = ["2025::" + str(i) for i in df2025.index]
df2026["source_record_id"] = ["2026::" + str(i) for i in df2026.index]




## Why fuzzy matching is not the first step

After the exact joins above, we are left with a much smaller unresolved pool.

That is the right time for fuzzy matching:

- fewer rows to compare,
- fewer false positives,
- and easier review.

Fuzzy matching is used as a **fallback**, not as the default merge strategy.


## Step 2: Introduction to fuzzy matching with RapidFuzz

Exact matching asks:

> Are these two strings exactly the same?

Fuzzy matching asks:

> How similar are these two strings?

This is useful when two records could refer to the same building, but the text is presented slightly differently. These differeces may appear because of naming convention where the data was sourced, but can also be caused by mistakes in data entry. Completing a normalization step attempts to resolve naming discrepencies, but it may not catch all.

For example:

- `"100 Main Street"` vs. `"100 MAIN ST."`
- `"Oak Tower"` vs. `"Oak TWR"`
- `"Main Plaza"` vs. `"Main Plaza Office"`

RapidFuzz gives us similarity scores from **0 to 100**:

- **100** = very similar or identical
- **0** = not similar
- values in between = partial similarity

For datasets like addresses, we started with exact matching. Now we'll use RapidFuzz as a fallback after exact matching has already reduced the number of unresolved records.

### What the different RapidFuzz scores mean

RapidFuzz has several scoring functions. They all compare two strings, but they are useful in slightly different situations.

| Scorer | What it is useful for |
|---|---|
| `fuzz.ratio()` | General character-level similarity |
| `fuzz.partial_ratio()` | One string may be contained inside another |
| `fuzz.token_sort_ratio()` | Same words, but possibly in a different order |
| `fuzz.token_set_ratio()` | One string may have extra words, but the core words overlap |
| `fuzz.WRatio()` | A weighted combination that often works well as a general-purpose scorer |

For building data, `token_set_ratio()` and `WRatio()` are often useful because building names and addresses can have extra words, abbreviations, or slightly different ordering.

### RapidFuzz example using one target address

Here we use one target string:

`100 Main Street`

Then we compare it against several small variations to investigate how the scores behave:

- all caps,
- punctuation differences,
- missing spaces,
- abbreviated `Street` to `St`,
- abbreviated `Street` to `St.`,
- and clearly different addresses that should not be treated as matches.

In [74]:
# Import RapidFuzz tools used in this example.
# fuzz contains scoring functions, process contains search helpers,
# and utils.default_process applies basic text preprocessing.
from rapidfuzz import fuzz, process, utils

# This is the address we want to compare against several variations.
target = "100 Main Street"

address_variations = [
    # Addresses that could plausibly refer to the target.
    "100 MAIN STREET",
    "10.0 MAIN STREET",
    "100Main Street",
    "100MainStreet",
    "100 Main St",
    "100 Main St.",

    # Addresses that should not be treated as the same location.
    "414 Bane Street",
    "212 B Baker Street"
]

# Store one dictionary of scores per comparison string.
score_examples = []

for comparison in address_variations:
    score_examples.append({
        "target": target,
        "comparison": comparison,

        # Basic ratio without preprocessing.
        "ratio": fuzz.ratio(target, comparison),

        # Basic ratio with RapidFuzz's default string cleanup.
        "ratio_processed": fuzz.ratio(
            target,
            comparison,
            processor=utils.default_process
        ),

        # Useful when one string is mostly contained inside the other.
        "partial_ratio": fuzz.partial_ratio(
            target,
            comparison,
            processor=utils.default_process
        ),

        # Useful when the same words appear in a different order.
        "token_sort_ratio": fuzz.token_sort_ratio(
            target,
            comparison,
            processor=utils.default_process
        ),

        # Useful when the strings share core words but one has extras.
        "token_set_ratio": fuzz.token_set_ratio(
            target,
            comparison,
            processor=utils.default_process
        ),

        # General-purpose weighted score.
        "WRatio": fuzz.WRatio(
            target,
            comparison,
            processor=utils.default_process
        )
    })

score_examples = pd.DataFrame(score_examples)
display(score_examples)


,target,comparison,ratio,ratio_processed,partial_ratio,token_sort_ratio,token_set_ratio,WRatio
0,100 Main Street,100 MAIN STREET,46.666667,100.000000,100.000000,100.000000,100.000000,100.000000
1,100 Main Street,10.0 MAIN STREET,45.161290,96.774194,93.333333,90.322581,90.322581,96.774194
2,100 Main Street,100Main Street,96.551724,96.551724,92.857143,96.551724,96.551724,96.551724
3,100 Main Street,100MainStreet,92.857143,92.857143,84.615385,92.857143,92.857143,92.857143
4,100 Main Street,100 Main St,84.615385,84.615385,100.000000,84.615385,84.615385,84.615385
5,100 Main Street,100 Main St.,81.481481,84.615385,100.000000,84.615385,84.615385,84.615385
6,100 Main Street,414 Bane Street,73.333333,73.333333,75.862069,73.333333,73.333333,73.333333
7,100 Main Street,212 B Baker Street,60.606061,60.606061,64.285714,60.606061,60.606061,60.606061


The first several comparison strings could refer to the same address in a real dataset. The output shows that scores can vary depending on the scorer.
A basic character-level ratio may penalize capitalization, punctuation, or spacing more strongly than a token-based scorer. In contrast, `token_set_ratio()` and `WRatio()` can be more forgiving when the important address tokens still overlap. The all caps `100 MAIN STREET` ratio is the one of the lowests scores out of all comparisons, even the clearly wrong adresses! But scores exceptinally well across the other fuzzy methods. 

Note that all the 'correct' addresses range from ~80% to 100%, while the clearly incorrect dip into the 70 and 60%. 

The key point is that fuzzy scores are evidence. A high score suggests that a pair deserves attention, but the pair should still be reviewed before accepted as a true match.


The fuzzy WRatio is a good general schema, and is what we will use going forward for this tutorial. But the other options could be even better for your data! 

### Searching one value against many possible values

The `process.extract()` function compares one query string to a list of possible choices.

This is useful when we have one unresolved 2025 building and want to search across many unresolved 2026 buildings.

The result gives us:

1. the possible match,
2. the similarity score,
3. the index position of the matched choice.

In a real merge workflow, we usually do not accept the fuzzy match automatically. Instead, we use the score to create a matches to review.

In [75]:
# Find the single best candidate for one query address.
best_match = process.extractOne(
    "100 Main Street",
    ["100 MAIN ST.", "200 River Road", "3300 Oak Avenue", "910 West Campus Drive"],
    scorer=fuzz.WRatio,
    processor=utils.default_process
)

# The output includes the best matching string, its score, and its index position.
best_match


('100 MAIN ST.', 84.61538461538461, 0)

### Prepare unresolved records for fuzzy matching

Now we will apply the Fuzzy Matching concept to the unresolved 2025 and 2026 building records.

Before scoring, we create normalized text fields. This helps reduce differences caused by capitalization, punctuation, or common address abbreviations.

The goal of this cell is to create `candidate_pairs`, where each unresolved 2025 record is paired with possible unresolved 2026 records.

Because we already completed normalization in Tutorial 1, we can use the normalized outputs from Tutorial 1 instead of redoing that work here. If starting with fuzzy matching normalization will need to be done.

## Step 2: Load outputs of tutorial 1

Now we will complete the fuzzy matching pipeline with tutorial 1 outputs. 

We will use a simple weighted score for the matching:

- address similarity carries most of the weight
- name similarity adds support
- same city/state can be treated as soft confirmation


In [76]:
from pathlib import Path
import pandas as pd

# Setup filepath to your directory and load the unresolved records saved from Tutorial 1.
TUTORIAL1_OUTPUT_DIR = Path.cwd() / "tutorial_output_csv"

df2025 = pd.read_csv(TUTORIAL1_OUTPUT_DIR / "df2025.csv")
df2026 = pd.read_csv(TUTORIAL1_OUTPUT_DIR / "df2026.csv")
exact_by_benchmark = pd.read_csv(TUTORIAL1_OUTPUT_DIR / "exact_by_benchmark.csv")
exact_by_address = pd.read_csv(TUTORIAL1_OUTPUT_DIR / "exact_by_address.csv")
unmatched_2025 = pd.read_csv(TUTORIAL1_OUTPUT_DIR / "unmatched_2025.csv")
unmatched_2026 = pd.read_csv(TUTORIAL1_OUTPUT_DIR / "unmatched_2026.csv")

print("Still unmatched in 2025:", len(unmatched_2025))
print("Still unmatched in 2026:", len(unmatched_2026))

display(unmatched_2025.head())
display(unmatched_2026.head())

Still unmatched in 2025: 1213
Still unmatched in 2026: 123


,benchmark_id,building_name,address,city,state,zip,reporting_year,source_record_id,benchmark_id_clean,name_norm,address_norm,city_norm,state_norm,zip_norm,addr_zip_key,name_addr_zip_key
0,01-CM-01410,NaN,1420 W 13th Ter,Kansas City,Missouri,64101,2025,2025::1,01CM01410,NaN,1420 W 13TH TER,KANSAS CITY,MISSOURI,64101,1420 W 13TH TER|64101,NaN
1,06-IS-01140,NaN,1117 E 9th St,Kansas City,Missouri,64106,2025,2025::2,06IS01140,NaN,1117 E 9TH ST,KANSAS CITY,MISSOURI,64106,1117 E 9TH ST|64106,NaN
2,01-IN-00720,Fern Expo,751 Wyoming St,Kansas City,Missouri,64101,2025,2025::4,01IN00720,FERN EXPO,751 WYOMING ST,KANSAS CITY,MISSOURI,64101,751 WYOMING ST|64101,FERN EXPO|751 WYOMING ST|64101
3,01-IN-00830,Faultless Brands,1100 W 8th St,Kansas City,Missouri,64101,2025,2025::5,01IN00830,FAULTLESS BRANDS,1100 W 8TH ST,KANSAS CITY,MISSOURI,64101,1100 W 8TH ST|64101,FAULTLESS BRANDS|1100 W 8TH ST|64101
4,01-IN-00925,Bemis,937 Wyoming St,Kansas City,Missouri,64101,2025,2025::6,01IN00925,BEMIS,937 WYOMING ST,KANSAS CITY,MISSOURI,64101,937 WYOMING ST|64101,BEMIS|937 WYOMING ST|64101


,benchmark_id,building_name,address,city,state,zip,reporting_year,source_record_id,benchmark_id_clean,name_norm,address_norm,city_norm,state_norm,zip_norm,addr_zip_key,name_addr_zip_key
0,Not Available,Golden Oaks Apartments,4319 North Holmes,Kansas City,Missouri,64116,2026,2026::4,NOTAVAILABLE,GOLDEN OAKS APARTMENTS,4319 NORTH HOLMES,KANSAS CITY,MISSOURI,64116,4319 NORTH HOLMES|64116,GOLDEN OAKS APARTMENTS|4319 NORTH HOLMES|64116
1,53CM09730,Wyndham Garden Kansas City Airport,7600 NW 97th Terrace,Kansas City,Missouri,64153,2026,2026::8,53CM09730,WYNDHAM GARDEN KANSAS CITY AIRPORT,7600 NW 97TH TERRACE,KANSAS CITY,MISSOURI,64153,7600 NW 97TH TERRACE|64153,WYNDHAM GARDEN KANSAS CITY AIRPORT|7600 NW 97T...
2,Not Available,86300-01 - 10711 N Congress Street - Kansas City,10711 N Congress Street,Kansas City,Missouri,64153,2026,2026::9,NOTAVAILABLE,86300 01 10711 N CONGRESS STREET KANSAS CITY,10711 N CONGRESS ST,KANSAS CITY,MISSOURI,64153,10711 N CONGRESS ST|64153,86300 01 10711 N CONGRESS STREET KANSAS CITY|1...
3,Not Available,KC Public Library Garage,106 W 10th,Kansas City,Missouri,64105,2026,2026::11,NOTAVAILABLE,KC PUBLIC LIBRARY GARAGE,106 W 10TH,KANSAS CITY,MISSOURI,64105,106 W 10TH|64105,KC PUBLIC LIBRARY GARAGE|106 W 10TH|64105
4,Not Available,Prime Investments,4700 Deramus,Kansas City,Missouri,64120,2026,2026::22,NOTAVAILABLE,PRIME INVESTMENTS,4700 DERAMUS,KANSAS CITY,MISSOURI,64120,4700 DERAMUS|64120,PRIME INVESTMENTS|4700 DERAMUS|64120


## Step 3.1: Build fuzzy candidates with blocking

We do not want compare every unresolved row to every other unresolved row, if a dataset was extremely large this would be more computationally expensive and take a long time.

Instead, we first **block** the comparison space using a practical geographic key such as ZIP. Blocking keeps the fuzzy stage smaller and more realistic for bigger datasets.

For example, if two buildings are in different ZIP codes, they will not be the same property, so we do not compare them in this simple version of the workflow.


In [77]:
# Build candidate pairs by matching only records with the same normalized ZIP code.
# # This code completes blocking: it limits which rows are compared to each other, not explicitly neccesary with smaller datasets
candidate_pairs = unmatched_2025.merge(
    unmatched_2026,
    on="zip_norm",
    how="inner",
    suffixes=("_2025", "_2026")
)

print("Candidate pairs after ZIP blocking:", len(candidate_pairs))

# Preview the fields most useful for checking whether a pair looks reasonable.
display(candidate_pairs[[
    "source_record_id_2025", "building_name_2025", "address_2025",
    "source_record_id_2026", "building_name_2026", "address_2026", "zip_norm"
]])


Candidate pairs after ZIP blocking: 5851


,source_record_id_2025,building_name_2025,address_2025,source_record_id_2026,building_name_2026,address_2026,zip_norm
0,2025::1,NaN,1420 W 13th Ter,2026::206,1405 St. Louis Ave,1405 St. Louis Avenue,64101
1,2025::2,NaN,1117 E 9th St,2026::273,The Commerce Bank Building,1000 Walnut,64106
2,2025::2,NaN,1117 E 9th St,2026::276,Tower Garage,910 Walnut,64106
3,2025::2,NaN,1117 E 9th St,2026::287,1102 Grand KCMO,1102 Grand,64106
4,2025::2,NaN,1117 E 9th St,2026::622,Center for Medical Education and Innovation,540 Highland Ave,64106
...,...,...,...,...,...,...,...
5846,2025::1795,NaN,8101 NW Barrybrooke Dr #8,2026::97,Line Creek Elementary,5801 NW Waukomis,64151
5847,2025::1795,NaN,8101 NW Barrybrooke Dr #8,2026::101,Southeast Elementary,5704 NW Northwood,64151
5848,2025::1795,NaN,8101 NW Barrybrooke Dr #8,2026::106,Chinn Elementary,7100 NW Chatham Rd,64151
5849,2025::1795,NaN,8101 NW Barrybrooke Dr #8,2026::109,English Landing Elementary,6500 NW Klamm Drive,64151


## Step 3.2: Score the fuzzy candidates

Now we score each candidate pair.

For teaching, we use a simple weighted score:

- address similarity carries most of the weight,
- name similarity adds support,
- same city adds a small bonus.

The final `match_score` is not meant to be perfect. It is a ranking tool that helps us decide which candidate pairs should be reviewed first.


In [78]:
# Score address similarity for each candidate pair.
# token_set_ratio is useful when one address has extra tokens or abbreviations.
candidate_pairs["address_score"] = candidate_pairs.apply(
    lambda r: fuzz.token_set_ratio(str(r["address_norm_2025"]), str(r["address_norm_2026"])),
    axis=1
)

# Score building-name similarity for each candidate pair.
candidate_pairs["name_score"] = candidate_pairs.apply(
    lambda r: fuzz.token_set_ratio(str(r["name_norm_2025"]), str(r["name_norm_2026"])),
    axis=1
)

# Add a small bonus when the normalized city matches across years.
candidate_pairs["city_match_bonus"] = np.where(
    candidate_pairs["city_norm_2025"] == candidate_pairs["city_norm_2026"], 5, 0
)

# Combine the component scores into one teaching-friendly match score.
# Address similarity gets the most weight because addresses are usually stronger identifiers.
candidate_pairs["match_score"] = (
    0.7 * candidate_pairs["address_score"]
    + 0.3 * candidate_pairs["name_score"]
    + candidate_pairs["city_match_bonus"]
)

# Sort from strongest to weakest candidate pair.
candidate_pairs = candidate_pairs.sort_values("match_score", ascending=False)

# Display the evidence used to evaluate each candidate pair.
display(candidate_pairs[[
    "source_record_id_2025", "source_record_id_2026",
    "building_name_2025", "building_name_2026",
    "address_2025", "address_2026",
    "address_score", "name_score", "city_match_bonus", "match_score"
]])


,source_record_id_2025,source_record_id_2026,building_name_2025,building_name_2026,address_2025,address_2026,address_score,name_score,city_match_bonus,match_score
304,2025::112,2026::287,1102 Grand KCMO,1102 Grand KCMO,1102 Grand Blvd,1102 Grand,100.000000,100.0,5,105.000000
277,2025::101,2026::276,Tower Garage,Tower Garage,910 Walnut St,910 Walnut,100.000000,100.0,5,105.000000
378,2025::151,2026::652,Two Pershing Square,Two Pershing Square,2300 Main St,2300 Main,100.000000,100.0,5,105.000000
381,2025::152,2026::188,Blue Cross Blue Shield of Kansas City-2301,Blue Cross Blue Shield of Kansas City-2301,2301 Main St,2301 main,100.000000,100.0,5,105.000000
893,2025::336,2026::282,The Summit Nursing Home,The Summit Nursing Home,3660 Summit St,3660 Summit,100.000000,100.0,5,105.000000
...,...,...,...,...,...,...,...,...,...,...
4993,2025::1587,2026::282,NaN,The Summit Nursing Home,3527 Broadway Blvd,3660 Summit,13.793103,0.0,5,14.655172
1205,2025::396,2026::243,NaN,Plaza 209,612 W 47th St,209 Emanuel Cleaver II Boulevard,13.333333,0.0,5,14.333333
3468,2025::1178,2026::243,NaN,Plaza 209,605 W 47th St,209 Emanuel Cleaver II Boulevard,13.333333,0.0,5,14.333333
3117,2025::1113,2026::652,NaN,Two Pershing Square,1919 Cherry St,2300 Main,8.695652,0.0,5,11.086957


## Step 4: Keep the best candidate per unresovlved row

In [79]:
# For each unresolved 2025 record, keep only the highest-scoring 2026 candidate.
best_fuzzy_per_2025 = (
    candidate_pairs
    .sort_values(["source_record_id_2025", "match_score"], ascending=[True, False])
    .drop_duplicates(subset=["source_record_id_2025"])
    .copy()
)

# Show the best candidate and its score for each unresolved 2025 record.
display(best_fuzzy_per_2025[[
    "source_record_id_2025", "source_record_id_2026",
    "building_name_2025", "building_name_2026",
    "address_2025", "address_2026",
    "match_score"
]])


,source_record_id_2025,source_record_id_2026,building_name_2025,building_name_2026,address_2025,address_2026,match_score
0,2025::1,2026::206,NaN,1405 St. Louis Ave,1420 W 13th Ter,1405 St. Louis Avenue,31.250000
11,2025::10,2026::206,NaN,1405 St. Louis Ave,1222 W 12th St,1405 St. Louis Avenue,32.096774
273,2025::100,2026::276,Evergy 801 Charlotte Emergency Response Center,Tower Garage,801 Charlotte St,910 Walnut,39.164456
2706,2025::1000,2026::280,NaN,Power & Light Apartments,1 W 12th St,106 West 14th Street,56.851852
2716,2025::1001,2026::280,KC LIVE 5TH PLAT,Power & Light Apartments,1 W 13th St,106 West 14th Street,66.325536
...,...,...,...,...,...,...,...
2651,2025::994,2026::159,The Landings at River Market,Second + Delaware,213 Delaware St,122 Delaware,70.767442
2660,2025::995,2026::133,West 7th Street Lofts,Fountain Lofts,318 W 7th St,408 W 8th Street,73.289474
2669,2025::996,2026::32,Quality Hill Towers,Mainmark Building,817 Jefferson St,1627 Main St.,50.000000
2687,2025::998,2026::530,Kansas City Club,11th and Wyandotte Garage,1228 Baltimore Ave,1020 Baltimore,67.099682


## Step 5: Build a review scheme

To make reviewing more managable we can split the fuzzy results into score bands:

- **high confidence** → can often be accepted quickly
- **medium confidence** → review recommended
- **low confidence** → hold back or inspect manually

This keeps the workflow conservative even when fuzzy logic is used.


In [80]:
# Convert numeric match scores into review categories.
best_fuzzy_per_2025["score_band"] = pd.cut(
    best_fuzzy_per_2025["match_score"],
    # Here we setup initial thresholds of 80%, 90%, and 100%
    bins=[-np.inf, 80, 90, 100],
    labels=["review", "likely", "strong"]
)

# Count how many candidate matches fall into each score band.
display(
    best_fuzzy_per_2025["score_band"]
    .value_counts(dropna=False)
    .rename_axis("score_band")
    .reset_index(name="count")
)

# Keep all non-strong candidates in a review queue.
# These are the pairs that need closer human inspection before acceptance.
review_queue = best_fuzzy_per_2025[best_fuzzy_per_2025["score_band"] != "strong"].copy()

# Display the review queue with the most relevant comparison fields.
display(review_queue[[
    "source_record_id_2025", "source_record_id_2026",
    "building_name_2025", "building_name_2026",
    "address_2025", "address_2026",
    "match_score", "score_band"
]])


,score_band,count
0,review,941
1,NaN,33
2,strong,22
3,likely,16


,source_record_id_2025,source_record_id_2026,building_name_2025,building_name_2026,address_2025,address_2026,match_score,score_band
0,2025::1,2026::206,NaN,1405 St. Louis Ave,1420 W 13th Ter,1405 St. Louis Avenue,31.250000,review
11,2025::10,2026::206,NaN,1405 St. Louis Ave,1222 W 12th St,1405 St. Louis Avenue,32.096774,review
273,2025::100,2026::276,Evergy 801 Charlotte Emergency Response Center,Tower Garage,801 Charlotte St,910 Walnut,39.164456,review
2706,2025::1000,2026::280,NaN,Power & Light Apartments,1 W 12th St,106 West 14th Street,56.851852,review
2716,2025::1001,2026::280,KC LIVE 5TH PLAT,Power & Light Apartments,1 W 13th St,106 West 14th Street,66.325536,review
...,...,...,...,...,...,...,...,...
2651,2025::994,2026::159,The Landings at River Market,Second + Delaware,213 Delaware St,122 Delaware,70.767442,review
2660,2025::995,2026::133,West 7th Street Lofts,Fountain Lofts,318 W 7th St,408 W 8th Street,73.289474,review
2669,2025::996,2026::32,Quality Hill Towers,Mainmark Building,817 Jefferson St,1627 Main St.,50.000000,review
2687,2025::998,2026::530,Kansas City Club,11th and Wyandotte Garage,1228 Baltimore Ave,1020 Baltimore,67.099682,review


In [81]:
# Score address similarity for each candidate pair.
# token_set_ratio is useful when one address has extra tokens or abbreviations.
candidate_pairs["address_score"] = candidate_pairs.apply(
    lambda r: fuzz.token_set_ratio(str(r["address_norm_2025"]), str(r["address_norm_2026"])),
    axis=1
)

# Score building-name similarity for each candidate pair.
candidate_pairs["name_score"] = candidate_pairs.apply(
    lambda r: fuzz.token_set_ratio(str(r["name_norm_2025"]), str(r["name_norm_2026"])),
    axis=1
)

# Add a small bonus when the normalized city matches across years.
candidate_pairs["city_match_bonus"] = np.where(
    candidate_pairs["city_norm_2025"] == candidate_pairs["city_norm_2026"], 5, 0
)

# Combine the component scores into one teaching-friendly match score.
# Address similarity gets the most weight because addresses are usually stronger identifiers.
candidate_pairs["match_score"] = (
    0.7 * candidate_pairs["address_score"]
    + 0.3 * candidate_pairs["name_score"]
    + candidate_pairs["city_match_bonus"]
)

# Sort from strongest to weakest candidate pair.
candidate_pairs = candidate_pairs.sort_values("match_score", ascending=False)

# Display the evidence used to evaluate each candidate pair.
display(candidate_pairs[[
    "source_record_id_2025", "source_record_id_2026",
    "building_name_2025", "building_name_2026",
    "address_2025", "address_2026",
    "address_score", "name_score", "city_match_bonus", "match_score"
]])


,source_record_id_2025,source_record_id_2026,building_name_2025,building_name_2026,address_2025,address_2026,address_score,name_score,city_match_bonus,match_score
304,2025::112,2026::287,1102 Grand KCMO,1102 Grand KCMO,1102 Grand Blvd,1102 Grand,100.000000,100.0,5,105.000000
277,2025::101,2026::276,Tower Garage,Tower Garage,910 Walnut St,910 Walnut,100.000000,100.0,5,105.000000
378,2025::151,2026::652,Two Pershing Square,Two Pershing Square,2300 Main St,2300 Main,100.000000,100.0,5,105.000000
381,2025::152,2026::188,Blue Cross Blue Shield of Kansas City-2301,Blue Cross Blue Shield of Kansas City-2301,2301 Main St,2301 main,100.000000,100.0,5,105.000000
893,2025::336,2026::282,The Summit Nursing Home,The Summit Nursing Home,3660 Summit St,3660 Summit,100.000000,100.0,5,105.000000
...,...,...,...,...,...,...,...,...,...,...
4993,2025::1587,2026::282,NaN,The Summit Nursing Home,3527 Broadway Blvd,3660 Summit,13.793103,0.0,5,14.655172
1205,2025::396,2026::243,NaN,Plaza 209,612 W 47th St,209 Emanuel Cleaver II Boulevard,13.333333,0.0,5,14.333333
3468,2025::1178,2026::243,NaN,Plaza 209,605 W 47th St,209 Emanuel Cleaver II Boulevard,13.333333,0.0,5,14.333333
3117,2025::1113,2026::652,NaN,Two Pershing Square,1919 Cherry St,2300 Main,8.695652,0.0,5,11.086957


## Step 5.1: Check the best candidate per unresolved 2025 row

This repeats the review pattern: sort by score, keep the best-scoring candidate per 2025 record, and inspect anything that is not clearly strong.

In [82]:
# For each unresolved 2025 record, keep only the highest-scoring 2026 candidate.
best_fuzzy_per_2025 = (
    candidate_pairs
    .sort_values(["source_record_id_2025", "match_score"], ascending=[True, False])
    .drop_duplicates(subset=["source_record_id_2025"])
    .copy()
)

# Show the best candidate and its score for each unresolved 2025 record.
display(best_fuzzy_per_2025[[
    "source_record_id_2025", "source_record_id_2026",
    "building_name_2025", "building_name_2026",
    "address_2025", "address_2026",
    "match_score"
]])


,source_record_id_2025,source_record_id_2026,building_name_2025,building_name_2026,address_2025,address_2026,match_score
0,2025::1,2026::206,NaN,1405 St. Louis Ave,1420 W 13th Ter,1405 St. Louis Avenue,31.250000
11,2025::10,2026::206,NaN,1405 St. Louis Ave,1222 W 12th St,1405 St. Louis Avenue,32.096774
273,2025::100,2026::276,Evergy 801 Charlotte Emergency Response Center,Tower Garage,801 Charlotte St,910 Walnut,39.164456
2706,2025::1000,2026::280,NaN,Power & Light Apartments,1 W 12th St,106 West 14th Street,56.851852
2716,2025::1001,2026::280,KC LIVE 5TH PLAT,Power & Light Apartments,1 W 13th St,106 West 14th Street,66.325536
...,...,...,...,...,...,...,...
2651,2025::994,2026::159,The Landings at River Market,Second + Delaware,213 Delaware St,122 Delaware,70.767442
2660,2025::995,2026::133,West 7th Street Lofts,Fountain Lofts,318 W 7th St,408 W 8th Street,73.289474
2669,2025::996,2026::32,Quality Hill Towers,Mainmark Building,817 Jefferson St,1627 Main St.,50.000000
2687,2025::998,2026::530,Kansas City Club,11th and Wyandotte Garage,1228 Baltimore Ave,1020 Baltimore,67.099682


We ended up with 1012 rows. Inspection on the `match_score` column in conjunction with the addresses between 2025 and 2026 shows most of these are not a match. 

## Step 5.2: Rebuild the review queue

We next split the fuzzy results into score bands and then inspect the candidates that are not strong enough to accept quickly.

- **high confidence** → can often be accepted quickly
- **medium confidence** → review recommended
- **low confidence** → hold back or inspect manually

This keeps the workflow consistent even when fuzzy logic is used. Even the **high confidence** should be reviewed and adjusted before accepted.

This section repeats the review-queue logic so the notebook can be rerun from this point after changes to the scoring cell.



In [83]:
# Convert numeric match scores into review categories.
best_fuzzy_per_2025["score_band"] = pd.cut(
    best_fuzzy_per_2025["match_score"],
    bins=[-np.inf, 80, 90, 100],
    labels=["review", "likely", "strong"]
)

# Count how many candidate matches fall into each score band.
display(
    best_fuzzy_per_2025["score_band"]
    .value_counts(dropna=False)
    .rename_axis("score_band")
    .reset_index(name="count")
)

# Keep all non-strong candidates in a review queue.
# These are the pairs that need closer human inspection before acceptance.
review_queue = best_fuzzy_per_2025[best_fuzzy_per_2025["score_band"] != "strong"].copy()

# Display the review queue with the most relevant comparison fields.
display(review_queue[[
    "source_record_id_2025", "source_record_id_2026",
    "building_name_2025", "building_name_2026",
    "address_2025", "address_2026",
    "match_score", "score_band"
]])


,score_band,count
0,review,941
1,NaN,33
2,strong,22
3,likely,16


,source_record_id_2025,source_record_id_2026,building_name_2025,building_name_2026,address_2025,address_2026,match_score,score_band
0,2025::1,2026::206,NaN,1405 St. Louis Ave,1420 W 13th Ter,1405 St. Louis Avenue,31.250000,review
11,2025::10,2026::206,NaN,1405 St. Louis Ave,1222 W 12th St,1405 St. Louis Avenue,32.096774,review
273,2025::100,2026::276,Evergy 801 Charlotte Emergency Response Center,Tower Garage,801 Charlotte St,910 Walnut,39.164456,review
2706,2025::1000,2026::280,NaN,Power & Light Apartments,1 W 12th St,106 West 14th Street,56.851852,review
2716,2025::1001,2026::280,KC LIVE 5TH PLAT,Power & Light Apartments,1 W 13th St,106 West 14th Street,66.325536,review
...,...,...,...,...,...,...,...,...
2651,2025::994,2026::159,The Landings at River Market,Second + Delaware,213 Delaware St,122 Delaware,70.767442,review
2660,2025::995,2026::133,West 7th Street Lofts,Fountain Lofts,318 W 7th St,408 W 8th Street,73.289474,review
2669,2025::996,2026::32,Quality Hill Towers,Mainmark Building,817 Jefferson St,1627 Main St.,50.000000,review
2687,2025::998,2026::530,Kansas City Club,11th and Wyandotte Garage,1228 Baltimore Ave,1020 Baltimore,67.099682,review


## Step 6: Complete the manual review

The fuzzy scores help prioritize records, but the final decision should come from manual review.

A reviewer should compare the building names, addresses, ZIP codes, and any additional context available from the source files. After review, each candidate can be labeled as accepted, rejected, or still unresolved.


-- Interactive review widget for fuzzy match inspection

- 1. Filter candidate matches by a minimum match_score threshold.
- 2. Inspect one candidate match at a time.
- 3. Mark each candidate as accepted, denied, or unsure.
- 4. Save notes for each reviewed pair.


Run the cell below to create an interactive review widget

In [84]:
# This cell provides a widget for the tutorial. The widget is not the focus so this has minimal comments. 
# Inspection can also be done manually through a created review csv and then imported

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Choose the columns we want to inspect during review.
review_columns = [
    "source_record_id_2025", "source_record_id_2026",
    "building_name_2025", "building_name_2026",
    "address_2025", "address_2026",
    "match_score", "score_band"
]

# Make a working copy so the original review_queue stays unchanged.
# The original index is saved so you can trace decisions back to the source data.
review_widget_df = (
    review_queue[review_columns]
    .copy()
    .reset_index()
    .rename(columns={"index": "original_row_index"})
)

# Add columns for human review decisions.
review_widget_df["review_decision"] = pd.NA
review_widget_df["review_notes"] = ""

# -----------------------
# Widget controls
# -----------------------

# Minimum score threshold used to filter visible candidate matches.
score_threshold = widgets.FloatText(
    value=80.0,
    description="Min score:",
    step=1.0
)

# Optional score band filter.
score_band_filter = widgets.Dropdown(
    options=["all"] + list(review_widget_df["score_band"].dropna().astype(str).unique()),
    value="all",
    description="Band:"
)

# Show all rows or only rows that have not yet been reviewed.
review_status_filter = widgets.Dropdown(
    options=["unreviewed", "all", "accepted", "denied", "unsure"],
    value="unreviewed",
    description="Status:"
)

notes_box = widgets.Textarea(
    value="",
    placeholder="Add notse the match decision for tracing...",
    description="Notes:",
    layout=widgets.Layout(width="700px", height="80px")
)

prev_button = widgets.Button(description="Previous")
next_button = widgets.Button(description="Next")
accept_button = widgets.Button(description="Accept match", button_style="success")
deny_button = widgets.Button(description="Deny match", button_style="danger")
unsure_button = widgets.Button(description="Mark unsure", button_style="warning")
save_note_button = widgets.Button(description="Save note")
reset_filters_button = widgets.Button(description="Reset filters")

match_output = widgets.Output()
summary_output = widgets.Output()

# Store the current position in the filtered review list.
state = {
    "position": 0,
    "current_row_index": None
}

# Helper functions

def get_filtered_indices():
    """
    Return the row indices from review_widget_df that match the current filters.
    """
    mask = review_widget_df["match_score"].fillna(-np.inf) >= score_threshold.value

    if score_band_filter.value != "all":
        mask &= review_widget_df["score_band"].astype(str).eq(score_band_filter.value)

    if review_status_filter.value == "unreviewed":
        mask &= review_widget_df["review_decision"].isna()
    elif review_status_filter.value != "all":
        mask &= review_widget_df["review_decision"].eq(review_status_filter.value)

    return review_widget_df.index[mask].tolist()


def save_current_note():
    """
    Save the current text box note into the working review DataFrame.
    """
    current_idx = state["current_row_index"]

    if current_idx is not None:
        review_widget_df.loc[current_idx, "review_notes"] = notes_box.value


def display_summary():
    """
    Display a small summary of review progress.
    """
    with summary_output:
        clear_output()

        decision_counts = (
            review_widget_df["review_decision"]
            .fillna("unreviewed")
            .value_counts()
            .rename_axis("review_decision")
            .reset_index(name="count")
        )

        display(decision_counts)


def display_current_match():
    """
    Display the currently selected candidate match.
    """
    filtered_indices = get_filtered_indices()

    with match_output:
        clear_output()

        if len(filtered_indices) == 0:
            state["current_row_index"] = None
            notes_box.value = ""

            display(HTML(
                "<b>No candidate matches found with the current filters.</b><br>"
                "Try lowering the score threshold or changing the review status filter."
            ))
            display_summary()
            return

        # Keep position within the valid range of the filtered results.
        state["position"] = state["position"] % len(filtered_indices)

        current_idx = filtered_indices[state["position"]]
        state["current_row_index"] = current_idx

        row = review_widget_df.loc[current_idx]

        # Load the saved note for this row into the notes widget.
        notes_box.value = row["review_notes"]

        # Basic progress display.
        display(HTML(
            f"""
            <h3>Candidate match {state["position"] + 1} of {len(filtered_indices)}</h3>
            <p>
                <b>Match score:</b> {row["match_score"]:.2f}
                &nbsp; | &nbsp;
                <b>Score band:</b> {row["score_band"]}
                &nbsp; | &nbsp;
                <b>Decision:</b> {row["review_decision"] if pd.notna(row["review_decision"]) else "unreviewed"}
            </p>
            """
        ))

        # Show the 2025 and 2026 records side by side.
        comparison_table = pd.DataFrame({
            "field": ["source_record_id", "building_name", "address"],
            "2025 record": [
                row["source_record_id_2025"],
                row["building_name_2025"],
                row["address_2025"]
            ],
            "2026 candidate": [
                row["source_record_id_2026"],
                row["building_name_2026"],
                row["address_2026"]
            ]
        })

        display(comparison_table)

    display_summary()


def move_next(_=None):
    """
    Save notes and move to the next visible candidate.
    """
    save_current_note()
    state["position"] += 1
    display_current_match()


def move_previous(_=None):
    """
    Save notes and move to the previous visible candidate.
    """
    save_current_note()
    state["position"] -= 1
    display_current_match()


def set_decision(decision):
    """
    Save a human decision for the current candidate match.
    """
    current_idx = state["current_row_index"]

    if current_idx is not None:
        review_widget_df.loc[current_idx, "review_decision"] = decision
        save_current_note()

    # Move forward after making a decision.
    state["position"] += 1
    display_current_match()


def refresh_filters(_=None):
    """
    Recalculate the visible candidate list after changing filters.
    """
    save_current_note()
    state["position"] = 0
    display_current_match()


def reset_filters(_=None):
    """
    Reset filters back to the default review view.
    """
    score_threshold.value = 80.0
    score_band_filter.value = "all"
    review_status_filter.value = "unreviewed"
    state["position"] = 0
    display_current_match()



# Connect buttons/actions

prev_button.on_click(move_previous)
next_button.on_click(move_next)
accept_button.on_click(lambda _: set_decision("accepted"))
deny_button.on_click(lambda _: set_decision("denied"))
unsure_button.on_click(lambda _: set_decision("unsure"))
save_note_button.on_click(lambda _: save_current_note())
reset_filters_button.on_click(reset_filters)

score_threshold.observe(refresh_filters, names="value")
score_band_filter.observe(refresh_filters, names="value")
review_status_filter.observe(refresh_filters, names="value")

# Display the widget

controls = widgets.VBox([
    widgets.HBox([score_threshold, score_band_filter, review_status_filter, reset_filters_button]),
    widgets.HBox([prev_button, next_button, accept_button, deny_button, unsure_button]),
    notes_box,
    save_note_button,
    summary_output,
    match_output
])

display(controls)

# Show the first candidate.
display_current_match()

We used fuzzy matching on address, but more fields could contain information. Complete the manual review and consider the address as well as building name 

## Step 7: Save and inspect missingness

After manual review, the next step is to save the reviewed matches for a final merged dataframe.

Useful outputs include:

- accepted fuzzy matches,
- rejected fuzzy candidates,
- records that could still need manual research,
- and a final missingness summary showing which fields are still incomplete.


In [85]:
# Use the same output folder as Tutorial 1 so all intermediate files stay together.
TUTORIAL_OUTPUT_DIR = Path.cwd() / "tutorial_output_csv"
TUTORIAL_OUTPUT_DIR.mkdir(exist_ok=True)

# The interactive review widget creates review_widget_df.
# If the widget output was already saved previously, this reloads it.
review_file = TUTORIAL_OUTPUT_DIR / "fuzzy_match_review_decisions.csv"

if "review_widget_df" in globals():
    reviewed_matches = review_widget_df.copy()
elif review_file.exists():
    reviewed_matches = pd.read_csv(review_file)
else:
    raise NameError(
        "Could not find review_widget_df or fuzzy_match_review_decisions.csv. "
        "Run the review widget first, then rerun this cell."
    )

# Standardize decision labels so filtering works even if capitalization or spacing varies.
reviewed_matches["review_decision"] = (
    reviewed_matches["review_decision"]
    .astype("string")
    .str.lower()
    .str.strip()
)

# Treat empty-looking decision values as missing.
reviewed_matches["review_decision"] = reviewed_matches["review_decision"].replace({
    "": pd.NA,
    "nan": pd.NA,
    "<na>": pd.NA,
    "none": pd.NA
})

# Separate reviewed candidates by their manual review decision.
accepted_fuzzy_matches = reviewed_matches[
    reviewed_matches["review_decision"] == "accepted"
].copy()

rejected_fuzzy_candidates = reviewed_matches[
    reviewed_matches["review_decision"] == "denied"
].copy()

# These candidate pairs either need more research or were never reviewed.
manual_research_candidate_pairs = reviewed_matches[
    reviewed_matches["review_decision"].isna()
    | reviewed_matches["review_decision"].eq("unsure")
].copy()

# A final merged file usually needs one-to-one matches.
# These checks identify accepted fuzzy matches where the same 2025 or 2026 record
# appears in more than one accepted pair.
accepted_conflict_mask = (
    accepted_fuzzy_matches.duplicated("source_record_id_2025", keep=False)
    | accepted_fuzzy_matches.duplicated("source_record_id_2026", keep=False)
)

accepted_match_conflicts = accepted_fuzzy_matches[accepted_conflict_mask].copy()

# Accepted matches without conflicts are the safest fuzzy matches to use for merging.
accepted_fuzzy_matches_for_merge = accepted_fuzzy_matches[~accepted_conflict_mask].copy()


def find_id_column(df, possible_names):
    """
    Find the first matching ID column name in a dataframe.

    This keeps the code flexible in case the saved unmatched files use slightly
    different ID column names.
    """
    for col in possible_names:
        if col in df.columns:
            return col

    raise ValueError(
        f"Could not find an ID column. Tried: {possible_names}"
    )


# Find the ID columns in the saved unmatched files.
id_col_2025 = find_id_column(
    unmatched_2025,
    ["source_record_id", "source_record_id_2025", "record_id", "id"]
)

id_col_2026 = find_id_column(
    unmatched_2026,
    ["source_record_id", "source_record_id_2026", "record_id", "id"]
)

# Pull the record IDs that were accepted as fuzzy matches.
accepted_2025_ids = set(
    accepted_fuzzy_matches["source_record_id_2025"]
    .dropna()
    .astype(str)
)

accepted_2026_ids = set(
    accepted_fuzzy_matches["source_record_id_2026"]
    .dropna()
    .astype(str)
)

# Remove accepted fuzzy matches from the still-unmatched pools.
remaining_unmatched_2025 = unmatched_2025[
    ~unmatched_2025[id_col_2025].astype(str).isin(accepted_2025_ids)
].copy()

remaining_unmatched_2026 = unmatched_2026[
    ~unmatched_2026[id_col_2026].astype(str).isin(accepted_2026_ids)
].copy()

# Combine remaining unmatched records into one file for follow-up research.
records_still_needing_research = pd.concat(
    [
        remaining_unmatched_2025.assign(source_year="2025"),
        remaining_unmatched_2026.assign(source_year="2026")
    ],
    ignore_index=True,
    sort=False
)

# Summarize how much the accepted fuzzy matches reduced the unmatched records.
match_progress_summary = pd.DataFrame({
    "item": [
        "Original unmatched 2025 records",
        "Original unmatched 2026 records",
        "Accepted fuzzy matches",
        "Rejected fuzzy candidates",
        "Unsure or unreviewed candidate pairs",
        "Remaining unmatched 2025 records after fuzzy review",
        "Remaining unmatched 2026 records after fuzzy review",
        "Accepted-match conflicts needing review"
    ],
    "count": [
        len(unmatched_2025),
        len(unmatched_2026),
        len(accepted_fuzzy_matches),
        len(rejected_fuzzy_candidates),
        len(manual_research_candidate_pairs),
        len(remaining_unmatched_2025),
        len(remaining_unmatched_2026),
        len(accepted_match_conflicts)
    ]
})

display(match_progress_summary)


def missingness_summary(df, dataset_name):
    """
    Summarize missing values by column for a dataframe.
    """
    return (
        df.isna()
        .sum()
        .rename("missing_count")
        .reset_index()
        .rename(columns={"index": "field"})
        .assign(
            dataset=dataset_name,
            total_rows=len(df),
            missing_percent=lambda x: np.where(
                x["total_rows"] > 0,
                round((x["missing_count"] / x["total_rows"]) * 100, 2),
                0
            )
        )
        [["dataset", "field", "total_rows", "missing_count", "missing_percent"]]
        .sort_values(["dataset", "missing_percent", "missing_count"], ascending=[True, False, False])
    )


# Check which fields are still incomplete after fuzzy review.
final_missingness_summary = pd.concat(
    [
        missingness_summary(remaining_unmatched_2025, "remaining_unmatched_2025_after_fuzzy"),
        missingness_summary(remaining_unmatched_2026, "remaining_unmatched_2026_after_fuzzy"),
        missingness_summary(accepted_fuzzy_matches, "accepted_fuzzy_matches")
    ],
    ignore_index=True
)

# optional display the final_missingness_summary
# display(final_missingness_summary)

# Optional saves.
# Uncomment any outputs you want to keep as CSV files.

# reviewed_matches.to_csv(
#     TUTORIAL_OUTPUT_DIR / "fuzzy_match_review_decisions.csv",
#     index=False
# )

# accepted_fuzzy_matches.to_csv(
#     TUTORIAL_OUTPUT_DIR / "accepted_fuzzy_matches.csv",
#     index=False
# )

# accepted_fuzzy_matches_for_merge.to_csv(
#     TUTORIAL_OUTPUT_DIR / "accepted_fuzzy_matches_for_merge.csv",
#     index=False
# )

# accepted_match_conflicts.to_csv(
#     TUTORIAL_OUTPUT_DIR / "accepted_match_conflicts.csv",
#     index=False
# )

# rejected_fuzzy_candidates.to_csv(
#     TUTORIAL_OUTPUT_DIR / "rejected_fuzzy_candidates.csv",
#     index=False
# )

# manual_research_candidate_pairs.to_csv(
#     TUTORIAL_OUTPUT_DIR / "manual_research_candidate_pairs.csv",
#     index=False
# )

# remaining_unmatched_2025.to_csv(
#     TUTORIAL_OUTPUT_DIR / "remaining_unmatched_2025_after_fuzzy.csv",
#     index=False
# )

# remaining_unmatched_2026.to_csv(
#     TUTORIAL_OUTPUT_DIR / "remaining_unmatched_2026_after_fuzzy.csv",
#     index=False
# )

# records_still_needing_research.to_csv(
#     TUTORIAL_OUTPUT_DIR / "records_still_needing_research.csv",
#     index=False
# )

# match_progress_summary.to_csv(
#     TUTORIAL_OUTPUT_DIR / "match_progress_summary.csv",
#     index=False
# )

final_missingness_summary.to_csv(
    TUTORIAL_OUTPUT_DIR / "final_missingness_summary.csv",
    index=False
)

print("Saved final missingness summary to:")
print(TUTORIAL_OUTPUT_DIR / "final_missingness_summary.csv")

,item,count
0,Original unmatched 2025 records,1213
1,Original unmatched 2026 records,123
2,Accepted fuzzy matches,0
3,Rejected fuzzy candidates,0
4,Unsure or unreviewed candidate pairs,990
5,Remaining unmatched 2025 records after fuzzy r...,1213
6,Remaining unmatched 2026 records after fuzzy r...,123
7,Accepted-match conflicts needing review,0


Saved final missingness summary to:
c:\Users\Joey\Desktop\github\KC_Cotality\curriculum\tutorial_output_csv\final_missingness_summary.csv


## Summary

The main purpose of fuzzy matching is to resolve discrepancies within the data. We did this with a set protocol that is reproducible:

1. Standardize the fields you will compare.  
2. Use exact keys first.  
3. Shrink the unresolved pool.  
4. Use fuzzy matching only on what exact logic could not resolve.  
5. Score candidate pairs in a transparent way.  
6. Keep a review queue instead of forcing uncertain matches.  
7. Save reviewed decisions so the workflow is reproducible.

Fuzzy matching is great because it helps find likely matches that exact logic misses. It is also risky if used too aggressively. The safest workflow is to use fuzzy scores as a guide for review rather than as automatic final decisions.


## Fuzzy matching understanding and further knowledge questions

Data quality and summary questions for the CCR created table.

---

### Question 1) Use the above widget and adjust the match_threshold. What threshold is likely to give best results?
 - #### Question 1.5) Consider: How will the false positive and false negative rates change as we sweep through the threshold?

---

### Question 2) Compare different fuzzy matching methods. Which method seems most appropriate for this dataset?
 - #### Question 2.5) Consider: `token_sort_ratio`, and `Wratio` Which method is more forgiving when words appear in a different order?

---

### Question 3) We used the address field for this fuzzy matching implementation, but we often have more fields like building name, latitude/longitude, and potentially others in our data.
 - #### How would we include multiple fields into a fuzzy matching protocol? Would including fields give a better outcome?

---